In [7]:
!pip install scikit-learn xgboost
!pip install \
    --extra-index-url=https://pypi.nvidia.com \
    "cudf-cu13==26.6.*" "dask-cudf-cu13==26.6.*" "cuml-cu13==26.6.*" \
    "cugraph-cu13==26.6.*" "nx-cugraph-cu13==26.6.*" "cuxfilter-cu13==26.6.*" \
    "cucim-cu13==26.6.*" "pylibraft-cu13==26.6.*" "raft-dask-cu13==26.6.*" \
    "cuvs-cu13==26.6.*" "nvforest-cu13==26.6.*" "nx-cugraph-cu13==26.6.*"

Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com


In [3]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.svm import SVC
from xgboost import XGBClassifier

random_seed = 2026

### The Plan
- Split trainval and test grouped by subject_ID (to prevent data leakage); stratify mind wandering
- KFold cross validation using trainval during hyperparameter gridsearch (I will try to prevent data leakage here as well); stratify mind wandering

## Load Data

In [5]:
# Load data
path = "data/preprocessed/data_channels_2000ms.pkl"
data = pd.read_pickle(path).reset_index(drop=True)
display(data.head())

# Split data
groups = data["subject"]
X = data[[*data.columns][2:]].astype(float)
y = data["is_mw"].astype(int)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=random_seed)

,subject,is_mw,delta_Fp1,delta_AF7,delta_AF3,delta_F1,delta_F3,delta_F5,delta_F7,delta_FT7,...,gamma_CP4,gamma_CP2,gamma_P2,gamma_P4,gamma_P6,gamma_P8,gamma_P10,gamma_PO8,gamma_PO4,gamma_O2
0,s10014,0.0,0.003993,0.858909,0.406686,0.463510,0.770716,0.566798,1.920785,1.009465,...,-0.523725,-0.274626,-1.234105,-0.499914,-0.041446,0.217153,0.981542,0.524016,-0.161951,-0.012756
1,s10014,0.0,-0.164858,0.480567,-0.086676,-0.446350,-0.030338,0.563824,1.295781,0.791540,...,-0.404339,-0.578598,0.370978,-0.126248,-0.329209,0.031090,0.393750,-0.215081,-0.017976,-0.519309
2,s10014,0.0,-0.935185,-0.042313,-0.482141,0.560272,1.442766,0.411945,0.848820,0.717990,...,-0.635014,-0.798848,0.900752,-0.442205,-0.033748,0.400009,0.440025,-0.248922,-0.035075,-0.560225
3,s10014,0.0,-0.680763,0.213668,-0.548231,-0.605734,-0.647036,-0.324204,0.359699,0.421622,...,-0.532251,-0.556259,-0.305672,-0.165480,0.199191,0.499053,1.495401,0.624022,-0.058816,0.004764
4,s10014,0.0,-0.326949,-0.808375,-0.392057,-0.063663,0.098948,-0.223435,0.243728,0.380588,...,-0.929942,-0.834456,-0.734891,-0.823642,-0.534384,0.258278,1.670351,0.157279,-0.171800,0.007077


## Training
### Logistic Regression

In [6]:
def train_logistic_regression(X, y, groups, parameters):
    cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=random_seed)

    lr = LogisticRegression(random_state=random_seed)
    clf = GridSearchCV(lr, parameters, cv=cv, scoring="roc_auc") 

    clf.fit(X,y,groups=groups)
    
    return clf

### SVM with radial basis function

In [7]:
def train_svc(X, y, groups, parameters):
    cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=random_seed)

    svc = SVC(probability = True,random_state=random_seed)
    clf = GridSearchCV(svc, parameters, cv=cv, scoring="roc_auc") 

    clf.fit(X,y,groups=groups)
    
    return clf

### XGBoost

In [8]:
def train_xgboost(X, y, groups, parameters):
    cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=random_seed)

    xgb = XGBClassifier(scale_pos_weight=len(y_trainval[y_trainval==0])/len(y_trainval[y_trainval==1]), random_state=random_seed, eval_metric="auc")
    clf = GridSearchCV(xgb, parameters, cv=cv, scoring="roc_auc") 

    clf.fit(X,y,groups=groups)
    
    return clf

### Training

In [17]:
lr_params = { # Why these?
    'C': [1, 10, 100], 
    'class_weight': ["balanced"],
    'solver': ["lbfgs", "liblinear"],
    'max_iter': [750,1250],
    'warm_start': [True, False]
}

svm_params = { # Why these?
    'C': [1, 10, 100],
    'gamma': ["auto","scale", 0.01, 0.1]
}

xgb_params = { # Why these?
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100, 200],
    'max_depth': [5, 10],
    'reg_alpha': [1, 5]
}

In [ ]:

for fold, (train_idxs, test_idxs) in enumerate(cv.split(X, y, groups)):
    X_trainval, y_trainval = X.iloc[train_idxs].reset_index(drop=True), y[train_idxs].reset_index(drop=True)
    X_test, y_test = X.iloc[test_idxs].reset_index(drop=True), y[test_idxs].reset_index(drop=True)
    groups_train = groups[train_idxs]
    groups_test = groups[test_idxs]

    print(f"Fold {fold}")
    best_logreg_clf = train_logistic_regression(X_trainval, y_trainval, groups_train, parameters=lr_params)
    perf_auc = roc_auc_score(y_test, best_logreg_clf.predict_proba(X_test)[:,1])
    print(f"ROC-AUC (LogisticRegression({best_logreg_clf.best_params_})): {perf_auc}")

    best_svm_clf = train_svc(X_trainval, y_trainval, groups_train, parameters=svm_params)
    perf_auc = roc_auc_score(y_test, best_svm_clf.decision_function(X_test))
    print(f"ROC-AUC (SVC({best_svm_clf.best_params_})): {perf_auc}")

    best_xgb_clf = train_xgboost(X_trainval, y_trainval, groups_train, parameters=xgb_params)
    perf_auc = roc_auc_score(y_test, best_xgb_clf.predict_proba(X_test)[:,1])
    print(f"ROC-AUC (XGBoost({best_xgb_clf.best_params_})): {perf_auc}")
    # print()
    # print("Fold :", fold)
    # print("TRAIN POSITIVE RATIO:", y[train_idxs].mean())
    # print("TEST POSITIVE RATIO :", y[test_idxs].mean())
    # print(f"TRAIN PERCENTAGE    : {len(train_idxs) / len(X) * 100:.4}%")
    # print("TRAIN GROUPS        :", set(groups[train_idxs]))
    # print("TEST GROUPS         :", set(groups[test_idxs]))


Fold 0
